# Sundew SDK Demo - Cloud Validation

This notebook validates the Sundew Core SDK in Google Colab without requiring physical hardware.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oluwafemidiakhoa/sundew_algorithms/blob/main/notebooks/Sundew_SDK_Demo.ipynb)

## What This Tests
- ✅ SDK installation and dependencies
- ✅ IPC protobuf bindings generation
- ✅ Adaptive gating controller
- ✅ TCP/socket transport layer
- ✅ Metrics tracking and telemetry
- ✅ Full test suite (12 tests)

In [ ]:
# Cell 1: Setup Environment
import sys
import os

# Clone repository
if not os.path.exists("sundew_algorithms"):
    !git clone https://github.com/oluwafemidiakhoa/sundew_algorithms.git
    %cd sundew_algorithms
else:
    %cd sundew_algorithms
    !git pull origin main

# Install dependencies
!pip install -q numpy pandas grpcio grpcio-tools protobuf

print("✅ Environment setup complete")

In [ ]:
# Cell 2: Generate IPC Bindings
!python tools/generate_ipc_bindings.py

# Verify bindings
if os.path.exists("src/sundew_ipc_v1_pb2.py"):
    print("\n✅ IPC bindings generated successfully")
else:
    print("\n❌ Binding generation failed")

In [ ]:
# Cell 3: Run IPC Demo
print("Running IPC Demo...\n")
!python examples/ipc_demo.py

In [ ]:
# Cell 4: Run SDK Test Suite
!pip install -q pytest pytest-cov hypothesis
!pytest tests/test_ipc*.py tests/test_grpc*.py -v --tb=short

In [ ]:
# Cell 5: Interactive SDK Testing
from sundew_core_sdk import SDKConfig, AdaptiveGateController, MetricsTracker
from sundew_core_sdk.ipc.adapter import IPCAdapter
from sundew_ipc_v1_pb2 import ScoreEvent, FeatureKV

# Initialize SDK
config = SDKConfig(target_activation=0.15, gate_temperature=0.08)
controller = AdaptiveGateController(config)
controller.load_native()

adapter = IPCAdapter(controller=controller, tracker=MetricsTracker())

# Create test event
event = ScoreEvent(
    sequence=1,
    features=[
        FeatureKV(key="glucose_mgdl", value=150.0),
        FeatureKV(key="heart_rate", value=75.0),
    ]
)

# Process event
decision = adapter.handle_score_event(event)
print(f"Gate Decision: {decision.should_activate}")
print(f"Confidence: {decision.confidence}")

# Get metrics
snapshot = adapter.tracker.snapshot()
print(f"\nActivation Rate: {snapshot.activation_rate:.2%}")
print(f"Samples Processed: {snapshot.samples}")

In [ ]:
# Cell 6: Compare Presets
from sundew.config_presets import get_preset

presets = ["tuned_v2", "aggressive", "conservative", "custom_breast_probe"]

print("Preset Comparison:")
print("-" * 60)

for preset_name in presets:
    preset = get_preset(preset_name)
    print(f"\n{preset_name}:")
    print(f"  Activation Threshold: {preset.activation_threshold}")
    print(f"  Target Activation Rate: {preset.target_activation_rate}")
    print(f"  Energy Pressure: {preset.energy_pressure}")
    print(f"  Gate Temperature: {preset.gate_temperature}")

In [ ]:
# Cell 7: Activation Pattern Visualization
!pip install -q matplotlib

import matplotlib.pyplot as plt
import numpy as np

# Simulate activation pattern
np.random.seed(42)
events = 100
activations = []

for i in range(events):
    event = ScoreEvent(
        sequence=i,
        features=[FeatureKV(key="signal", value=np.random.randn())]
    )
    decision = adapter.handle_score_event(event)
    activations.append(1 if decision.should_activate else 0)

# Plot
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(activations, 'o-', markersize=3)
plt.title('Activation Pattern')
plt.xlabel('Event Number')
plt.ylabel('Activated (1=Yes, 0=No)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
cumulative_activation = np.cumsum(activations) / np.arange(1, events + 1)
plt.plot(cumulative_activation)
plt.title('Cumulative Activation Rate')
plt.xlabel('Event Number')
plt.ylabel('Activation Rate')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Activation Rate: {np.mean(activations):.2%}")
print(f"Energy Savings Estimate: {(1 - np.mean(activations)):.2%}")

## ✅ Validation Complete!

If all cells ran successfully:
- SDK is correctly installed
- IPC bindings are generated
- All tests pass
- Gating controller works
- Metrics tracking is functional

### Next Steps:
1. Test on your Surface laptop (see `docs/SURFACE_TESTING_GUIDE.md`)
2. Deploy to Raspberry Pi or Jetson for real hardware validation
3. Integrate power sensors (INA219/INA3221)

### Resources:
- GitHub: https://github.com/oluwafemidiakhoa/sundew_algorithms
- SDK Docs: `docs/sdk/README.md`
- Phase 1 Plan: `docs/phase1_sundew_core_sdk_plan.md`